# 💊 [Project] 경구 알약 객체 탐지 모델 개발 및 아키텍처별 성능 비교

1. 프로젝트 개요 (Project Overview)
배경: 경구 조제 약물 오복용 방지를 위한 자동 식별 및 객체 탐지 시스템 구축

목표: Faster R-CNN, YOLOv11, YOLOv26의 아키텍처를 활용한 성능 비교 및 알약 도메인 최적화

팀 구성: 용준, 도영, 설아, 다현, 예람

진행 방식: 데이터 엔지니어링(2인), 모델 설계 및 실험(각 3인) 전문 분업 체계

성명,역할 (Role),주요 수행 작업 및 핵심 기여
---

용준,Project Manager: "프로젝트 총괄, Faster R-CNN 설계 및 하이퍼파라미터 최적화, Backbone Unfreezing 전략 수립"

도영,Model Specialist: "YOLO26m + EfficientNet-B3 하이브리드 설계, TTA/후처리 튜닝으로 Kaggle 0.987 달성"

다현,Experiment Lead: "YOLOv11x 검증, 각인 보호를 위한 Flip(반전) 증강 차단 전략 수립, 1024 고해상도 학습 적용"

설아,System Engineer: "데이터 정합성 검증 및 동적 경로 매핑 구현, GPU 메모리 오프로딩 전략 수립, mAP 평가 자동화"

예람,Data Engineer: "AIHub 데이터 변환 시스템 구축, 1,000장 이상 대규모 데이터셋 통합 관리, 팀 협업용 시각화 도구 배포"

## 경로 설정 및 라이브러리 호출

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

In [2]:
import sys
!{sys.executable} -m pip install -q ultralytics pycocotools pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.9 MB/s eta 0:00:0000:01


In [ ]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################

import os
import gc
import json
import yaml
import shutil
import random
import itertools
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.ops import nms
import torch.optim as optim

import torchvision.transforms as T
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from ultralytics import YOLO

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
DEVICE: cuda
TRAIN_JSON_PATH exists: True
TEST_JSON_PATH exists : True
TRAIN_IMG_DIR exists  : True
TEST_IMG_DIR exists   : True
train image 수 : 1489
train ann 수   : 4526
test image 수  : 0
test ann 수    : 0
   ann_id  coco_image_id                                          file_name  \
0       1              1  K-001900-016551-024850-027926_0_2_0_2_90_000_2...   
1       2              2  K-001900-016551-024850-027926_0_2_0_2_70_000_2...   
2       3              3  K-001900-016551-024850-027926_0_2_0_2_75_000_2...   
3       6              2  K-001900-016551-024850-027926_0_2_0_2_70_000_2...   
4       7              2  K-001900-016551-024850-027926_0_2_0_2_70_000_2...   

       

In [ ]:


############################################################
# 0. 경로 / 설정
############################################################

def normalize_path(path):
    return unicodedata.normalize("NFC", path).strip()

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

extract_path = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset")

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH  = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR   = os.path.join(extract_path, "train_images")
TEST_IMG_DIR    = os.path.join(extract_path, "test_images")

WORK_DIR   = os.path.join(extract_path, "yolo_effb3_pipeline")
YOLO_ROOT  = os.path.join(WORK_DIR, "yolo_oneclass")
MODEL_DIR  = os.path.join(WORK_DIR, "saved_models")
PRED_DIR   = os.path.join(WORK_DIR, "predictions")
EVAL_DIR   = os.path.join(WORK_DIR, "eval")

ensure_dir(WORK_DIR)
ensure_dir(YOLO_ROOT)
ensure_dir(MODEL_DIR)
ensure_dir(PRED_DIR)
ensure_dir(EVAL_DIR)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
YOLO_DEVICE = 0 if torch.cuda.is_available() else "cpu"

# 환경에 맞게 수정
YOLO_MODEL_NAME = "/content/drive/MyDrive/data/초급_프로젝트/dataset/models/yolo26m.pt"   # 예: "/content/drive/MyDrive/weights/yolo26m.pt"

# detector
YOLO_IMGSZ = 640
YOLO_EPOCHS = 30
YOLO_BATCH = 8
YOLO_WORKERS = 0   # 중요: Colab thread 에러 방지

# classifier
CLS_IMG_SIZE = 300
CLS_EPOCHS = 10
CLS_BATCH = 16
CLS_LR = 3e-4
CLS_NUM_WORKERS = 0   # 중요

# crop
CROP_PAD_RATIO = 0.08

# 추론 / 제출
DET_CONF = 0.05
DET_IOU = 0.5
MAX_SUB_PER_IMAGE = 4
SUBMISSION_CATEGORY_OFFSET = 1   # 제출 사이트가 +1 요구하면 1, 아니면 0

# 가능하면 기존 98점 best detector 재사용
USE_EXISTING_YOLO_BEST = True
EXISTING_YOLO_BEST_PT = os.path.join(WORK_DIR, "yolo26m_oneclass", "weights", "best.pt")
# 직접 따로 저장한 best.pt 경로가 있으면 위 경로 대신 그 경로 넣어도 됨

# classifier 학습 조금 강화
CLS_EPOCHS = 12
CLS_LR = 2e-4

# detector train은 기존 유지, inference만 강화
YOLO_INFER_IMGSZ_CANDIDATES = [
    [640],
    [768],
    [640, 768],   # 가장 추천
]

DET_CONF_CANDIDATES = [0.03, 0.05]
PAD_RATIO_CANDIDATES = [0.08, 0.10, 0.12]
BETA_CANDIDATES = [1.0, 1.2, 1.4, 1.6]
TOPK_CANDIDATES = [4]

ALPHA_FIXED = 1.0
TTA_MAX_DET = 100

BEST_CFG_JSON = os.path.join(EVAL_DIR, "best_cfg_99try.json")
TEMP_JSON_PATH = os.path.join(MODEL_DIR, "cls_temperature_99try.json")
TUNING_CSV_PATH = os.path.join(EVAL_DIR, "tuning_results_99try.csv")

print("DEVICE:", DEVICE)
print("TRAIN_JSON_PATH exists:", os.path.exists(TRAIN_JSON_PATH))
print("TEST_JSON_PATH exists :", os.path.exists(TEST_JSON_PATH))
print("TRAIN_IMG_DIR exists  :", os.path.exists(TRAIN_IMG_DIR))
print("TEST_IMG_DIR exists   :", os.path.exists(TEST_IMG_DIR))


## 2. 데이터 전처리 및 엔지니어링 (예람, 설아)

1. 이종 모델 대응 파이프라인: YOLO(상대 좌표)와 Faster R-CNN(절대 좌표) 포맷 변환 프로세스 자동화 

2. 방어적 프로그래밍 적용: 이미지-JSON 라벨 간 1:1 매칭 검증(os.path.exists)으로 데이터 유실 차단 

3. 데이터 규격화: 세션 재시작 시에도 일관된 ID를 유지하는 고정 딕셔너리 기반 매핑 테이블 구축

In [ ]:

############################################################
# 1. COCO JSON -> images_df / ann_df
############################################################

def load_coco_tables(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    image_columns = [
        "coco_image_id", "file_name", "image_stem",
        "image_path", "img_w", "img_h"
    ]

    ann_columns = [
        "ann_id", "coco_image_id", "file_name", "image_stem", "image_path",
        "img_w", "img_h", "category_id",
        "bbox_x", "bbox_y", "bbox_w", "bbox_h"
    ]

    image_records = []
    image_id_to_info = {}

    for img in data.get("images", []):
        coco_image_id = int(img["id"])
        file_name = img["file_name"]
        image_path = os.path.join(img_dir, file_name)

        if not os.path.exists(image_path):
            continue

        width = img.get("width", None)
        height = img.get("height", None)

        if width is None or height is None:
            with Image.open(image_path) as im:
                width, height = im.size

        rec = {
            "coco_image_id": coco_image_id,
            "file_name": file_name,
            "image_stem": os.path.splitext(file_name)[0],
            "image_path": image_path,
            "img_w": int(width),
            "img_h": int(height),
        }
        image_records.append(rec)
        image_id_to_info[coco_image_id] = rec

    images_df = pd.DataFrame(image_records, columns=image_columns)

    ann_records = []
    for ann in data.get("annotations", []):
        coco_image_id = int(ann["image_id"])

        if coco_image_id not in image_id_to_info:
            continue

        bbox = ann.get("bbox", None)
        if bbox is None or len(bbox) != 4:
            continue

        x, y, w, h = bbox
        info = image_id_to_info[coco_image_id]

        ann_records.append({
            "ann_id": int(ann.get("id", len(ann_records) + 1)),
            "coco_image_id": coco_image_id,
            "file_name": info["file_name"],
            "image_stem": info["image_stem"],
            "image_path": info["image_path"],
            "img_w": info["img_w"],
            "img_h": info["img_h"],
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    ann_df = pd.DataFrame(ann_records, columns=ann_columns)
    return images_df, ann_df, data



train_images_df, train_ann_df, train_raw_json = load_coco_tables(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
test_images_df, test_ann_df, test_raw_json = load_coco_tables(TEST_JSON_PATH, TEST_IMG_DIR)

print("train image 수 :", len(train_images_df))
print("train ann 수   :", len(train_ann_df))
print("test image 수  :", len(test_images_df))
print("test ann 수    :", len(test_ann_df))
print(train_ann_df.head())


In [ ]:

############################################################
# 2. bbox 정리 / split / category mapping
############################################################

def sanitize_ann_df(ann_df):
    bad_cols = [
        "ann_id", "file_name", "image_stem",
        "bbox_x", "bbox_y", "bbox_w", "bbox_h", "img_w", "img_h"
    ]

    if ann_df is None or len(ann_df) == 0:
        return ann_df.copy(), pd.DataFrame(columns=bad_cols)

    required_cols = ["bbox_x", "bbox_y", "bbox_w", "bbox_h", "img_w", "img_h"]
    missing = [c for c in required_cols if c not in ann_df.columns]
    if missing:
        raise KeyError(f"sanitize_ann_df에 필요한 컬럼이 없습니다: {missing}")

    df = ann_df.copy()
    for c in required_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    x1 = df["bbox_x"].to_numpy(dtype=float)
    y1 = df["bbox_y"].to_numpy(dtype=float)
    x2 = x1 + df["bbox_w"].to_numpy(dtype=float)
    y2 = y1 + df["bbox_h"].to_numpy(dtype=float)

    x1n = np.minimum(x1, x2)
    y1n = np.minimum(y1, y2)
    x2n = np.maximum(x1, x2)
    y2n = np.maximum(y1, y2)

    img_w = df["img_w"].to_numpy(dtype=float)
    img_h = df["img_h"].to_numpy(dtype=float)

    x1c = np.clip(x1n, 0, img_w)
    y1c = np.clip(y1n, 0, img_h)
    x2c = np.clip(x2n, 0, img_w)
    y2c = np.clip(y2n, 0, img_h)

    valid = (
        np.isfinite(x1c) & np.isfinite(y1c) &
        np.isfinite(x2c) & np.isfinite(y2c) &
        np.isfinite(img_w) & np.isfinite(img_h) &
        (x2c > x1c) & (y2c > y1c)
    )

    bad_df = df.loc[~valid, bad_cols].copy()

    clean_df = df.loc[valid].copy()
    clean_df["bbox_x"] = x1c[valid]
    clean_df["bbox_y"] = y1c[valid]
    clean_df["bbox_w"] = x2c[valid] - x1c[valid]
    clean_df["bbox_h"] = y2c[valid] - y1c[valid]

    return clean_df.reset_index(drop=True), bad_df.reset_index(drop=True)


train_ann_df, bad_train_ann_df = sanitize_ann_df(train_ann_df)

if len(test_ann_df) > 0:
    test_ann_df, bad_test_ann_df = sanitize_ann_df(test_ann_df)
else:
    bad_test_ann_df = pd.DataFrame()

print("정리 후 train ann 수:", len(train_ann_df))
print("제거된 잘못된 train ann 수:", len(bad_train_ann_df))
print("정리 후 test ann 수 :", len(test_ann_df))


In [4]:
############################################################
# 2. train/val split + classifier용 category 매핑
############################################################

def split_images(images_df, train_ratio=0.9, seed=42):
    stems = images_df["image_stem"].tolist()
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(stems), generator=g).tolist()
    split = int(len(stems) * train_ratio)

    train_stems = {stems[i] for i in perm[:split]}
    val_stems   = {stems[i] for i in perm[split:]}
    return train_stems, val_stems

def ensure_all_classes_in_train(ann_df, train_stems, val_stems):
    train_stems = set(train_stems)
    val_stems = set(val_stems)

    cats = sorted(ann_df["category_id"].astype(int).unique().tolist())

    for cat in cats:
        in_train = ann_df[
            (ann_df["image_stem"].isin(train_stems)) &
            (ann_df["category_id"].astype(int) == cat)
        ]
        if len(in_train) == 0:
            candidates = ann_df[
                (ann_df["image_stem"].isin(val_stems)) &
                (ann_df["category_id"].astype(int) == cat)
            ]["image_stem"].drop_duplicates().tolist()

            if len(candidates) > 0:
                chosen = candidates[0]
                val_stems.remove(chosen)
                train_stems.add(chosen)
                print(f"[split 보정] val -> train 이동: {chosen} (category={cat})")

    return train_stems, val_stems


train_stems, val_stems = split_images(train_images_df, train_ratio=0.9, seed=42)
train_stems, val_stems = ensure_all_classes_in_train(train_ann_df, train_stems, val_stems)

print("train image split:", len(train_stems))
print("val image split  :", len(val_stems))


# classifier용 category mapping
unique_cats = sorted(train_ann_df["category_id"].astype(int).unique().tolist())
orig2cls = {orig_cat: idx for idx, orig_cat in enumerate(unique_cats)}
cls2orig = {idx: orig_cat for orig_cat, idx in orig2cls.items()}

CLS_LABEL_MAP_PATH = os.path.join(MODEL_DIR, "cls_idx_to_orig_cat.json")
with open(CLS_LABEL_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump({str(k): int(v) for k, v in cls2orig.items()}, f, ensure_ascii=False, indent=2)

print("분류 클래스 수:", len(unique_cats))
print("label map 저장:", CLS_LABEL_MAP_PATH)



train image split: 1340
val image split  : 149
분류 클래스 수: 73
label map 저장: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/saved_models/cls_idx_to_orig_cat.json


## 데이터 변환

In [ ]:
############################################################
# 3. COCO -> YOLO one-class dataset export
############################################################

def copy_if_needed(src, dst):
    ensure_dir(os.path.dirname(dst))
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

def coco_xywh_to_yolo(x, y, w, h, img_w, img_h):
    xc = (x + w / 2.0) / img_w
    yc = (y + h / 2.0) / img_h
    wn = w / img_w
    hn = h / img_h
    return xc, yc, wn, hn

def export_yolo_oneclass_dataset(images_df, ann_df, train_stems, val_stems, out_root):
    for split_name in ["train", "val"]:
        ensure_dir(os.path.join(out_root, "images", split_name))
        ensure_dir(os.path.join(out_root, "labels", split_name))

    ann_groups = {stem: g.copy() for stem, g in ann_df.groupby("image_stem")}
    split_map = {"train": train_stems, "val": val_stems}

    for split_name, stems in split_map.items():
        sub_images = images_df[images_df["image_stem"].isin(stems)].copy()

        for _, row in tqdm(sub_images.iterrows(), total=len(sub_images), desc=f"YOLO export [{split_name}]"):
            image_stem = row["image_stem"]
            src_img = row["image_path"]
            dst_img = os.path.join(out_root, "images", split_name, row["file_name"])
            copy_if_needed(src_img, dst_img)

            label_path = os.path.join(out_root, "labels", split_name, f"{image_stem}.txt")
            img_w, img_h = int(row["img_w"]), int(row["img_h"])
            anns = ann_groups.get(image_stem, None)

            with open(label_path, "w", encoding="utf-8") as f:
                if anns is not None and len(anns) > 0:
                    for _, ann_row in anns.iterrows():
                        x, y, w, h = ann_row["bbox_x"], ann_row["bbox_y"], ann_row["bbox_w"], ann_row["bbox_h"]
                        xc, yc, wn, hn = coco_xywh_to_yolo(x, y, w, h, img_w, img_h)
                        f.write(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

    data_yaml = {
        "path": out_root,
        "train": "images/train",
        "val": "images/val",
        "names": ["drug"],
    }

    yaml_path = os.path.join(out_root, "data.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

    return yaml_path


YOLO_DATA_YAML = export_yolo_oneclass_dataset(
    images_df=train_images_df,
    ann_df=train_ann_df,
    train_stems=train_stems,
    val_stems=val_stems,
    out_root=YOLO_ROOT
)

print("YOLO data yaml:", YOLO_DATA_YAML)

YOLO export [train]:   0%|          | 0/1340 [00:00<?, ?it/s]

YOLO export [val]:   0%|          | 0/149 [00:00<?, ?it/s]

YOLO data yaml: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/yolo_oneclass/data.yaml
classifier train size: 4079
classifier val size  : 445


### 데이터셋 로드

In [ ]:
############################################################
# 4. EfficientNet-B3 classifier Dataset / DataLoader
############################################################

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

cls_train_tfms = T.Compose([
    T.Resize((CLS_IMG_SIZE, CLS_IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=8),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cls_val_tfms = T.Compose([
    T.Resize((CLS_IMG_SIZE, CLS_IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def safe_crop_xywh(image, x, y, w, h, pad_ratio=0.08):
    img_w, img_h = image.size

    x = float(x); y = float(y); w = float(w); h = float(h)
    if w <= 0 or h <= 0:
        return None

    x1 = x
    y1 = y
    x2 = x + w
    y2 = y + h

    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1

    pad_x = (x2 - x1) * pad_ratio
    pad_y = (y2 - y1) * pad_ratio

    left   = int(np.floor(x1 - pad_x))
    top    = int(np.floor(y1 - pad_y))
    right  = int(np.ceil(x2 + pad_x))
    bottom = int(np.ceil(y2 + pad_y))

    left   = max(0, min(left, img_w))
    top    = max(0, min(top, img_h))
    right  = max(0, min(right, img_w))
    bottom = max(0, min(bottom, img_h))

    if right <= left or bottom <= top:
        return None

    return image.crop((left, top, right, bottom))


def jitter_bbox_xywh(x, y, w, h, img_w, img_h,
                     shift_ratio=0.10,
                     scale_ratio=0.10):
    # detector가 약간 흔들린 박스도 classifier가 견디도록 학습용 bbox jitter
    cx = x + w / 2.0
    cy = y + h / 2.0

    cx += random.uniform(-shift_ratio, shift_ratio) * w
    cy += random.uniform(-shift_ratio, shift_ratio) * h

    w = w * random.uniform(1.0 - scale_ratio, 1.0 + scale_ratio)
    h = h * random.uniform(1.0 - scale_ratio, 1.0 + scale_ratio)

    x = max(0.0, cx - w / 2.0)
    y = max(0.0, cy - h / 2.0)
    w = min(float(img_w) - x, w)
    h = min(float(img_h) - y, h)

    if w <= 0 or h <= 0:
        return None

    return x, y, w, h


class CropClassificationDataset(Dataset):
    def __init__(self, ann_df, stems, orig2cls, transform=None, pad_ratio=0.08, train_mode=False):
        self.df = ann_df[ann_df["image_stem"].isin(stems)].reset_index(drop=True).copy()
        self.orig2cls = orig2cls
        self.transform = transform
        self.pad_ratio = pad_ratio
        self.train_mode = train_mode

        self.df = self.df[self.df["category_id"].astype(int).isin(self.orig2cls.keys())].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]

        with Image.open(img_path) as im:
            image = im.convert("RGB")

        x = float(row["bbox_x"])
        y = float(row["bbox_y"])
        w = float(row["bbox_w"])
        h = float(row["bbox_h"])
        img_w = float(row["img_w"])
        img_h = float(row["img_h"])

        if self.train_mode:
            jittered = jitter_bbox_xywh(x, y, w, h, img_w, img_h)
            if jittered is not None:
                x, y, w, h = jittered

        crop = safe_crop_xywh(
            image=image,
            x=x,
            y=y,
            w=w,
            h=h,
            pad_ratio=self.pad_ratio
        )

        if crop is None:
            crop = image.copy()

        label = self.orig2cls[int(row["category_id"])]

        if self.transform is not None:
            crop = self.transform(crop)

        return crop, label


train_cls_ds = CropClassificationDataset(
    ann_df=train_ann_df,
    stems=train_stems,
    orig2cls=orig2cls,
    transform=cls_train_tfms,
    pad_ratio=CROP_PAD_RATIO,
    train_mode=True
)

val_cls_ds = CropClassificationDataset(
    ann_df=train_ann_df,
    stems=val_stems,
    orig2cls=orig2cls,
    transform=cls_val_tfms,
    pad_ratio=CROP_PAD_RATIO,
    train_mode=False
)

# class imbalance 완화
train_labels = [orig2cls[int(c)] for c in train_cls_ds.df["category_id"].astype(int).tolist()]
class_counts = np.bincount(train_labels, minlength=len(unique_cats))
sample_weights = [1.0 / max(class_counts[label], 1) for label in train_labels]

train_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_cls_loader = DataLoader(
    train_cls_ds,
    batch_size=CLS_BATCH,
    sampler=train_sampler,
    num_workers=CLS_NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

val_cls_loader = DataLoader(
    val_cls_ds,
    batch_size=CLS_BATCH,
    shuffle=False,
    num_workers=CLS_NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("classifier train size:", len(train_cls_ds))
print("classifier val size  :", len(val_cls_ds))


## 모델 실험 및 아키텍처 비교 분석 (용준, 설아, 다현)

1. Faster R-CNN + EfficientNet-B3
성과: 안정적인 탐지 성능으로 약 97점대 확보 및 전체 학습/예측 흐름 완성
최적화: EfficientNet-B3 백본 채택 및 Full Fine-tuning을 통해 Validation Loss 0.16 달성


2. YOLOv11: 최신 SOTA 모델 검증
성과: Kaggle 0.97, Recall 0.998 달성. 소형 객체 탐지력 확인
전략: 각인 왜곡 방지를 위해 Flip(반전) 증강 차단 및 1024 고해상도 학습 진행


3. YOLO26m + EfficientNet-B3
성과: 단일 모델의 한계를 극복하고 탐지 성능과 분류 정확도를 동시에 잡아 팀 내 최상위 성능 도출
전략: 성능이 검증된 98점대 베이스라인 가중치를 보존하면서 후처리 및 제출 포맷(CSV)의 규격화에 집중하여 실수를 방지

## 🏆최종 모델: YOLO26m + EfficientNet-B3 하이브리드

In [6]:
############################################################
# 5-1. YOLOv26m one-class detector 학습 / 또는 기존 best 재사용
############################################################

if USE_EXISTING_YOLO_BEST and os.path.exists(EXISTING_YOLO_BEST_PT):
    YOLO_BEST_PT = EXISTING_YOLO_BEST_PT
    print("기존 98점 detector best 재사용:", YOLO_BEST_PT)
else:
    det_model = YOLO(YOLO_MODEL_NAME)

    det_model.train(
        data=YOLO_DATA_YAML,
        epochs=YOLO_EPOCHS,
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
        device=YOLO_DEVICE,
        project=WORK_DIR,
        name="yolo26m_oneclass",
        exist_ok=True,
        workers=YOLO_WORKERS,
        patience=10,
        verbose=True
    )

    YOLO_BEST_PT = None  # 5-2에서 찾음


############################################################
# 5-2. YOLO best weight 경로 찾기
############################################################

def resolve_yolo_best_path(work_dir, run_name="yolo26m_oneclass"):
    candidates = [
        os.path.join(work_dir, run_name, "weights", "best.pt"),
        os.path.join(work_dir, run_name, "weights", "last.pt"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"YOLO weight를 찾지 못했습니다. candidates={candidates}")

YOLO_BEST_PT = resolve_yolo_best_path(WORK_DIR, "yolo26m_oneclass")
print("YOLO_BEST_PT:", YOLO_BEST_PT)

############################################################
# 5-3. detector validation metric (선택)
############################################################

detector_for_val = YOLO(YOLO_BEST_PT)

det_metrics = detector_for_val.val(
    data=YOLO_DATA_YAML,
    imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH,
    device=YOLO_DEVICE,
    split="val",
    workers=YOLO_WORKERS,
    verbose=False
)

print("YOLO one-class detector 결과")
print("mAP50-95:", det_metrics.box.map)
print("mAP50   :", det_metrics.box.map50)
print("mAP75   :", det_metrics.box.map75)


############################################################
# 6-1. EfficientNet-B3 classifier 설정
############################################################

cls_model = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)

in_features = cls_model.classifier[1].in_features
cls_model.classifier[1] = nn.Linear(in_features, len(unique_cats))
cls_model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
optimizer = optim.AdamW(cls_model.parameters(), lr=CLS_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CLS_EPOCHS)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

CLS_BEST_PTH = os.path.join(MODEL_DIR, "efficientnet_b3_cls_best.pth")
best_val_acc = 0.0

print("classifier 준비 완료")
print("num_classes:", len(unique_cats))
print("save path:", CLS_BEST_PTH)

############################################################
# 6-2. EfficientNet-B3 classifier 학습
############################################################

for epoch in range(CLS_EPOCHS):
    # train
    cls_model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_cls_loader, desc=f"CLS Train [{epoch+1}/{CLS_EPOCHS}]"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = cls_model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / max(1, train_total)
    train_acc = train_correct / max(1, train_total)

    # val
    cls_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_cls_loader, desc=f"CLS Val [{epoch+1}/{CLS_EPOCHS}]"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = cls_model(images)
                loss = criterion(logits, labels)

            val_loss_sum += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / max(1, val_total)
    val_acc = val_correct / max(1, val_total)

    scheduler.step()

    print(
        f"[Epoch {epoch+1}/{CLS_EPOCHS}] "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(cls_model.state_dict(), CLS_BEST_PTH)
        print("✅ classifier best 저장:", CLS_BEST_PTH)

print("classifier 학습 완료")

############################################################
# 6-3. temperature scaling
############################################################

def collect_logits_and_labels(model, loader, device):
    model.eval()
    logits_list = []
    labels_list = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            logits_list.append(logits)
            labels_list.append(labels)

    logits = torch.cat(logits_list, dim=0)
    labels = torch.cat(labels_list, dim=0)
    return logits, labels


def fit_temperature(model, loader, device):
    logits, labels = collect_logits_and_labels(model, loader, device)

    temperature = nn.Parameter(torch.ones(1, device=device))
    nll_criterion = nn.CrossEntropyLoss()
    optimizer_t = optim.LBFGS([temperature], lr=0.01, max_iter=50)

    def closure():
        optimizer_t.zero_grad()
        loss = nll_criterion(logits / temperature.clamp(min=1e-3), labels)
        loss.backward()
        return loss

    optimizer_t.step(closure)
    return float(temperature.detach().clamp(min=1e-3).item())


cls_model.load_state_dict(torch.load(CLS_BEST_PTH, map_location=DEVICE))
cls_model.eval()

CLS_TEMPERATURE = fit_temperature(cls_model, val_cls_loader, DEVICE)
print("CLS_TEMPERATURE:", CLS_TEMPERATURE)

with open(TEMP_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({"temperature": CLS_TEMPERATURE}, f, ensure_ascii=False, indent=2)

print("temperature 저장:", TEMP_JSON_PATH)




기존 98점 detector best 재사용: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/yolo26m_oneclass/weights/best.pt
YOLO_BEST_PT: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/yolo26m_oneclass/weights/best.pt
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 3.3±0.8 MB/s, size: 1728.9 KB)
val: Scanning /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/yolo_oneclass/labels/val.cache... 149 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 27.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 9.7s/it 3:049.9s1s
                   all        149        445       0.77      0.998      0.824      0.819
Speed: 0.3ms preprocess, 7.2ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /con

100%|██████████| 47.2M/47.2M [00:00<00:00, 136MB/s] 


classifier 준비 완료
num_classes: 73
save path: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/saved_models/efficientnet_b3_cls_best.pth


/tmp/ipykernel_27563/846926120.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


CLS Train [1/12]:   0%|          | 0/255 [00:00<?, ?it/s]

...

CLS Val [11/12]:   0%|          | 0/28 [00:00<?, ?it/s]

[Epoch 11/12] train_loss=0.2796, train_acc=0.9980 | val_loss=0.3046, val_acc=0.9955


CLS Train [12/12]:   0%|          | 0/255 [00:00<?, ?it/s]

CLS Val [12/12]:   0%|          | 0/28 [00:00<?, ?it/s]

[Epoch 12/12] train_loss=0.2804, train_acc=0.9980 | val_loss=0.3050, val_acc=0.9955
classifier 학습 완료
CLS_TEMPERATURE: 0.9147979617118835
temperature 저장: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/saved_models/cls_temperature_99try.json


### 최적의 모델을 기반으로 학습

In [7]:
############################################################
# 7-1. detector/classifier load + 개선된 inference 함수 정의
############################################################

# detector load
detector = YOLO(YOLO_BEST_PT)

# classifier load
with open(CLS_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    cls2orig_loaded = {int(k): int(v) for k, v in json.load(f).items()}

classifier = efficientnet_b3(weights=None)
in_features = classifier.classifier[1].in_features
classifier.classifier[1] = nn.Linear(in_features, len(cls2orig_loaded))
classifier.load_state_dict(torch.load(CLS_BEST_PTH, map_location=DEVICE))
classifier.to(DEVICE)
classifier.eval()

if os.path.exists(TEMP_JSON_PATH):
    with open(TEMP_JSON_PATH, "r", encoding="utf-8") as f:
        CLS_TEMPERATURE = float(json.load(f)["temperature"])
else:
    CLS_TEMPERATURE = 1.0

print("CLS_TEMPERATURE:", CLS_TEMPERATURE)


@torch.no_grad()
def classify_crops_batch(pil_crops, classifier, transform, device, temperature=1.0):
    if len(pil_crops) == 0:
        return [], []

    x = torch.stack([transform(crop) for crop in pil_crops]).to(device)

    with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
        logits = classifier(x)
        logits = logits / max(float(temperature), 1e-6)
        probs = torch.softmax(logits, dim=1)

    pred_conf, pred_idx = probs.max(dim=1)
    return pred_idx.cpu().tolist(), pred_conf.cpu().tolist()


@torch.no_grad()
def get_detector_boxes_tta(
    img_path,
    detector,
    device,
    imgsz_list,
    conf=0.05,
    iou=0.5,
    yolo_device=0,
    max_det=100,
):
    all_boxes = []
    all_scores = []

    for imgsz in imgsz_list:
        results = detector.predict(
            source=img_path,
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            max_det=max_det,
            device=yolo_device,
            verbose=False
        )

        result = results[0]
        if result.boxes is None or len(result.boxes) == 0:
            continue

        boxes = result.boxes.xyxy.cpu()
        scores = result.boxes.conf.cpu()

        all_boxes.append(boxes)
        all_scores.append(scores)

    if len(all_boxes) == 0:
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32)

    all_boxes = torch.cat(all_boxes, dim=0)
    all_scores = torch.cat(all_scores, dim=0)

    keep = nms(all_boxes, all_scores, iou_threshold=0.5)
    all_boxes = all_boxes[keep].numpy()
    all_scores = all_scores[keep].numpy()

    order = np.argsort(-all_scores)
    all_boxes = all_boxes[order][:max_det]
    all_scores = all_scores[order][:max_det]

    return all_boxes, all_scores


@torch.no_grad()
def run_pipeline_on_image_v99(
    img_path,
    detector,
    classifier,
    cls_transform,
    cls2orig,
    device,
    conf=0.05,
    iou=0.5,
    imgsz_list=(640, 768),
    yolo_device=0,
    crop_pad_ratio=0.08,
    alpha=1.0,
    beta=1.2,
    temperature=1.0,
    max_det=100,
):
    image = Image.open(img_path).convert("RGB")

    boxes_xyxy, det_scores = get_detector_boxes_tta(
        img_path=img_path,
        detector=detector,
        device=device,
        imgsz_list=imgsz_list,
        conf=conf,
        iou=iou,
        yolo_device=yolo_device,
        max_det=max_det,
    )

    outputs = []
    if len(boxes_xyxy) == 0:
        return outputs

    pil_crops = []
    meta = []

    for box, det_score in zip(boxes_xyxy, det_scores):
        x1, y1, x2, y2 = map(float, box)
        w = x2 - x1
        h = y2 - y1

        crop = safe_crop_xywh(image, x1, y1, w, h, pad_ratio=crop_pad_ratio)
        if crop is None:
            continue

        pil_crops.append(crop)
        meta.append((x1, y1, x2, y2, w, h, float(det_score)))

    if len(pil_crops) == 0:
        return outputs

    pred_idxs, cls_scores = classify_crops_batch(
        pil_crops=pil_crops,
        classifier=classifier,
        transform=cls_transform,
        device=device,
        temperature=temperature,
    )

    for (x1, y1, x2, y2, w, h, det_score), pred_idx, cls_score in zip(meta, pred_idxs, cls_scores):
        orig_cat = cls2orig[int(pred_idx)]
        final_score = float((det_score ** alpha) * (float(cls_score) ** beta))

        outputs.append({
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "w": w, "h": h,
            "det_score": float(det_score),
            "cls_idx": int(pred_idx),
            "orig_cat": int(orig_cat),
            "cls_score": float(cls_score),
            "final_score": final_score,
        })

    outputs = sorted(outputs, key=lambda x: x["final_score"], reverse=True)
    return outputs




CLS_TEMPERATURE: 0.9147979617118835


In [9]:
############################################################
# 7-2. test 추론 -> submission 생성
# ※ 8번 tuning 후 다시 실행할 것
############################################################

if not os.path.exists(BEST_CFG_JSON):
    raise FileNotFoundError(f"best config가 없습니다. 먼저 8번을 실행하세요: {BEST_CFG_JSON}")

with open(BEST_CFG_JSON, "r", encoding="utf-8") as f:
    best_cfg = json.load(f)

print("best_cfg:", best_cfg)

test_files = sorted([
    f for f in os.listdir(TEST_IMG_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp"))
])

rows_submit = []
annotation_id = 1

for f in tqdm(test_files, desc="Test inference"):
    img_path = os.path.join(TEST_IMG_DIR, f)
    image_stem = os.path.splitext(f)[0]

    preds = run_pipeline_on_image_v99(
        img_path=img_path,
        detector=detector,
        classifier=classifier,
        cls_transform=cls_val_tfms,
        cls2orig=cls2orig_loaded,
        device=DEVICE,
        conf=best_cfg["det_conf"],
        iou=DET_IOU,
        imgsz_list=best_cfg["imgsz_list"],
        yolo_device=YOLO_DEVICE,
        crop_pad_ratio=best_cfg["pad_ratio"],
        alpha=best_cfg["alpha"],
        beta=best_cfg["beta"],
        temperature=CLS_TEMPERATURE,
        max_det=TTA_MAX_DET,
    )

    preds = preds[:best_cfg["topk"]]

    for p in preds:
        submit_cat = int(p["orig_cat"]) + SUBMISSION_CATEGORY_OFFSET

        rows_submit.append({
            "annotation_id": annotation_id,
            "image_id": image_stem,
            "category_id": submit_cat,
            "bbox_x": p["x1"],
            "bbox_y": p["y1"],
            "bbox_w": p["w"],
            "bbox_h": p["h"],
            "score": p["final_score"],
        })
        annotation_id += 1

df_sub = pd.DataFrame(rows_submit, columns=[
    "annotation_id", "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
])

SUBMISSION_CSV_PATH = os.path.join(PRED_DIR, "final_submission_yolo26m_effb3_99try.csv")
df_sub.to_csv(SUBMISSION_CSV_PATH, index=False)

print("✅ submission 저장:", SUBMISSION_CSV_PATH)
print("총 제출 row 수:", len(df_sub))
print(df_sub.head())


best_cfg: {'imgsz_list': [768], 'det_conf': 0.03, 'pad_ratio': 0.08, 'alpha': 1.0, 'beta': 1.0, 'topk': 4}


Test inference:   0%|          | 0/843 [00:00<?, ?it/s]

/tmp/ipykernel_27563/1522666209.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


✅ submission 저장: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/predictions/final_submission_yolo26m_effb3_99try.csv
총 제출 row 수: 3232
   annotation_id image_id  category_id      bbox_x      bbox_y      bbox_w  \
0              1        1        24850  174.513336  740.693054  177.935577   
1              2        1         1900  158.938385  251.934814  203.901428   
2              3        1        27926  595.853027  675.323853  264.215210   
3              4        1        16551  556.500549   72.089539  397.184143   
4              5       10         1900  644.093445  843.523499  187.888123   

       bbox_h     score  
0  292.224548  0.787384  
1  123.452332  0.773624  
2  477.438721  0.746104  
3  405.273926  0.732528  
4  190.558044  0.794278  


In [8]:
############################################################
# 8. val split 기준 best setting 탐색 (coco mAP)
############################################################

def build_val_coco_gt(train_raw_json, val_stems):
    val_stems = set(val_stems)

    images = []
    annotations = []
    valid_image_ids = set()

    for img in train_raw_json.get("images", []):
        stem = os.path.splitext(img["file_name"])[0]
        if stem in val_stems:
            images.append(img)
            valid_image_ids.add(int(img["id"]))

    for ann in train_raw_json.get("annotations", []):
        if int(ann["image_id"]) in valid_image_ids:
            annotations.append(ann)

    val_gt = {
        "images": images,
        "annotations": annotations,
        "categories": train_raw_json.get("categories", [])
    }
    return val_gt


VAL_GT_JSON_PATH = os.path.join(EVAL_DIR, "val_gt.json")
val_gt = build_val_coco_gt(train_raw_json, val_stems)

with open(VAL_GT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(val_gt, f, ensure_ascii=False)

print("VAL_GT_JSON_PATH:", VAL_GT_JSON_PATH)
print("val images:", len(val_gt["images"]))
print("val anns  :", len(val_gt["annotations"]))

val_stem_to_gtid = {
    os.path.splitext(img["file_name"])[0]: int(img["id"])
    for img in val_gt["images"]
}

val_image_rows = train_images_df[train_images_df["image_stem"].isin(val_stems)].copy()

best_cfg = None
best_map = -1.0
tuning_logs = []

grid = list(itertools.product(
    YOLO_INFER_IMGSZ_CANDIDATES,
    DET_CONF_CANDIDATES,
    PAD_RATIO_CANDIDATES,
    BETA_CANDIDATES,
    TOPK_CANDIDATES,
))

print("tuning cases:", len(grid))

for imgsz_list, det_conf, pad_ratio, beta, topk in grid:
    rows_eval = []

    for _, row in tqdm(
        val_image_rows.iterrows(),
        total=len(val_image_rows),
        desc=f"TUNE imgsz={imgsz_list}, conf={det_conf}, pad={pad_ratio}, beta={beta}"
    ):
        img_path = row["image_path"]
        image_stem = row["image_stem"]

        preds = run_pipeline_on_image_v99(
            img_path=img_path,
            detector=detector,
            classifier=classifier,
            cls_transform=cls_val_tfms,
            cls2orig=cls2orig_loaded,
            device=DEVICE,
            conf=det_conf,
            iou=DET_IOU,
            imgsz_list=imgsz_list,
            yolo_device=YOLO_DEVICE,
            crop_pad_ratio=pad_ratio,
            alpha=ALPHA_FIXED,
            beta=beta,
            temperature=CLS_TEMPERATURE,
            max_det=TTA_MAX_DET,
        )

        preds = preds[:topk]

        for p in preds:
            rows_eval.append({
                "image_id": val_stem_to_gtid[image_stem],
                "category_id": int(p["orig_cat"]),
                "bbox": [p["x1"], p["y1"], p["w"], p["h"]],
                "score": float(p["final_score"]),
            })

    val_pred_json_path = os.path.join(
        EVAL_DIR,
        f"val_pred_imgsz{'-'.join(map(str, imgsz_list))}_conf{det_conf}_pad{pad_ratio}_beta{beta}_top{topk}.json"
    )

    with open(val_pred_json_path, "w", encoding="utf-8") as f:
        json.dump(rows_eval, f, ensure_ascii=False)

    coco_gt = COCO(VAL_GT_JSON_PATH)
    if len(rows_eval) == 0:
        map_score = 0.0
    else:
        coco_dt = coco_gt.loadRes(val_pred_json_path)
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
        map_score = float(coco_eval.stats[0])

    tuning_logs.append({
        "imgsz_list": str(imgsz_list),
        "det_conf": det_conf,
        "pad_ratio": pad_ratio,
        "beta": beta,
        "topk": topk,
        "map5095": map_score,
    })

    print(
        f"[TUNE] imgsz={imgsz_list}, conf={det_conf}, pad={pad_ratio}, "
        f"beta={beta}, topk={topk} -> mAP={map_score:.6f}"
    )

    if map_score > best_map:
        best_map = map_score
        best_cfg = {
            "imgsz_list": list(imgsz_list),
            "det_conf": float(det_conf),
            "pad_ratio": float(pad_ratio),
            "alpha": float(ALPHA_FIXED),
            "beta": float(beta),
            "topk": int(topk),
        }

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tuning_df = pd.DataFrame(tuning_logs)
tuning_df.to_csv(TUNING_CSV_PATH, index=False, encoding="utf-8-sig")

with open(BEST_CFG_JSON, "w", encoding="utf-8") as f:
    json.dump(best_cfg, f, ensure_ascii=False, indent=2)

print("best_map:", best_map)
print("best_cfg:", best_cfg)
print("tuning csv:", TUNING_CSV_PATH)
print("best cfg json:", BEST_CFG_JSON)

display(tuning_df.sort_values("map5095", ascending=False).head(10))


VAL_GT_JSON_PATH: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_effb3_pipeline/eval/val_gt.json
val images: 149
val anns  : 445
tuning cases: 72


TUNE imgsz=[640], conf=0.03, pad=0.08, beta=1.0:   0%|          | 0/149 [00:00<?, ?it/s]

/tmp/ipykernel_27563/1522666209.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.38s).
Accumulating evaluation results...
DONE (t=0.30s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.870
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.877
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.877
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.870
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.991
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.991
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

TUNE imgsz=[640], conf=0.03, pad=0.08, beta=1.2:   0%|          | 0/149 [00:00<?, ?it/s]

...

TUNE imgsz=[640, 768], conf=0.05, pad=0.12, beta=1.4:   0%|          | 0/149 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.18s).
Accumulating evaluation results...
DONE (t=0.29s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.865
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.874
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.874
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.865
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.989
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.989
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

TUNE imgsz=[640, 768], conf=0.05, pad=0.12, beta=1.6:   0%|          | 0/149 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.18s).
Accumulating evaluation results...
DONE (t=0.29s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.863
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.872
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.872
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.863
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.989
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.989
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

,imgsz_list,det_conf,pad_ratio,beta,topk,map5095
24,[768],0.03,0.08,1.0,4,0.877714
36,[768],0.05,0.08,1.0,4,0.877714
37,[768],0.05,0.08,1.2,4,0.877121
25,[768],0.03,0.08,1.2,4,0.877121
26,[768],0.03,0.08,1.4,4,0.876921
38,[768],0.05,0.08,1.4,4,0.876921
39,[768],0.05,0.08,1.6,4,0.876817
27,[768],0.03,0.08,1.6,4,0.876817
32,[768],0.03,0.12,1.0,4,0.876735
44,[768],0.05,0.12,1.0,4,0.876735


# 결론 및 시사점
## 최종 아키텍처 결정
정밀도 중심의 Faster R-CNN과 성능 중심의 YOLOv11 실험을 거쳐, 실제 배포 효율성이 가장 뛰어난 YOLO26m을 최종 모델로 확정


## 기술적 통찰
1. 도메인 특성(알약 각인, 소형 객체)에 따른 증강 전략의 중요성과 모델 규모에 따른 효율성 차이를 정량적으로 검증함

2. 98점대 이상의 고득점에 도달한 후에는 무리한 구조 변경보다, 검증된 가중치(Weight)를 보존하며 추론 전략을 개선하는 것이 안정적인 성과를 내는 핵심임을 체감


## 비판적 성찰
1) 추론 속도와 복잡도의 트레이드오프
TTA(Test Time Augmentation)까지 적용된 상태라 모바일 기기나 저사양 에지(Edge) 디바이스에서 구동하기엔 연산 부담이 큼.

2) 클래스 불균형과 롱테일 문제 (Class Imbalance)
특정 의약품은 데이터가 매우 부족한 Long-tail 분포를 가졌는데, 이를 해결 못함

